# Graph Registry Smoke Test

This notebook tests the integrated backend graphs from `app/flows/graph_registry.py`.
It is designed to run in mock mode (`force_mock_mode=True`) for stable local checks.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "app").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from app.core.config import Settings
from app.flows.graph_registry import build_graph_registry

settings = Settings(force_mock_mode=True)
registry = build_graph_registry(settings)
registry

In [ ]:
from app.schemas.copy import CopyGenerateRequest, CopyGenerateResponse, Objective, Style

copy_request = CopyGenerateRequest(
    product_name="Copyjoe",
    target_audience="Performance marketer",
    pain_point="Ad CTR is dropping and copy iteration is too slow",
    differentiator="Evidence-backed generation from RAG + web context",
    tone="Trustworthy",
    objective=Objective.click,
    styles=[Style.head, Style.body, Style.cta],
    channel="Landing page",
    language="en",
    web_search_mode=False,
    use_rag=False,
    top_k=5,
)

copy_output, copy_sources = registry.copy.run(copy_request)
copy_response = CopyGenerateResponse(
    head=copy_output.head,
    body=copy_output.body,
    cta=copy_output.cta,
    slogan=copy_output.slogan,
    sns=copy_output.sns,
    description=copy_output.description,
    storyboard_outline=copy_output.storyboard_outline,
    rationale=copy_output.rationale,
    sources=copy_sources,
)

assert copy_response.head
assert copy_response.cta
copy_response

In [ ]:
from app.schemas.copy import CopyLiteRequest

lite_payload = CopyLiteRequest(
    prompt="We need higher CTR copy for marketers with low ad performance.",
    styles=[Style.head, Style.body, Style.cta],
    use_rag=False,
    web_search_mode=False,
    top_k=5,
)

lite_response = await registry.copy_lite.run(lite_payload)
assert lite_response.result.head
assert lite_response.normalized_request.objective in {
    Objective.brand_memory,
    Objective.click,
    Objective.add_to_cart,
    Objective.consultation,
}
lite_response

In [ ]:
from io import BytesIO

from fastapi import UploadFile

upload = UploadFile(
    file=BytesIO(b"Copyjoe helps marketers improve CTR with evidence-backed copy generation."),
    filename="graph_registry_smoke.txt",
)
upload_result = await registry.file_upload.run([upload])
assert upload_result.success_count == 1, upload_result

doc_ids = [item.document_id for item in upload_result.files if item.success and item.document_id]
index_result = registry.rag.run_index(doc_ids, chunk_size=200, chunk_overlap=50)
search_result = registry.rag.run_search("improve CTR", top_k=3)
context_text, context_sources = registry.rag.run_build_context("improve CTR", top_k=3)

assert index_result.indexed_documents >= 1
assert len(search_result) >= 1
assert context_text
{
    "indexed_documents": index_result.indexed_documents,
    "search_hits": len(search_result),
    "context_sources": len(context_sources),
}

In [ ]:
thread = registry.history.create_thread("Graph registry notebook thread")
_ = registry.history.append_message(thread.thread_id, "user", "hello from notebook")
detail = registry.history.get_thread(thread.thread_id)
assert detail.thread.message_count == 1

guide = registry.meta.run()
assert len(guide.fields) >= 1

file_name, md_bytes = registry.export.run_markdown("graph_registry_notebook", copy_response)
md_preview = md_bytes.decode("utf-8")[:300]
assert file_name.endswith(".md")
assert "Copyjoe Export" in md_preview

{
    "thread_id": detail.thread.thread_id,
    "guide_fields": len(guide.fields),
    "export_file": file_name,
    "preview": md_preview,
}

## Optional web checks

If you have Tavily configured and network access:

```python
web_results = registry.web.run_search("performance copywriting", max_results=3, strict=True)
landing = await registry.web.run_analyze_landing_page("https://example.com")
```